# Step 6a — Apply soil constraint to the AMOC-collapse future scenarios

AND/weight-combines the per-crop GAEZ-formula soil-suitability layer from
`1_SVM/2_soil_suitability_gaez` (Module IV, same source now used by every `5.x)` validation
notebook -- changed 2026-10-14, was the older simple `1_SVM/2_soil_suitability` module) with
the climate-only Stage-4 predictions for the two AMOC-hosing methods this pipeline actually
targets: the **direct** bias-adjustment (EC-Earth3,
`4_SVM_predict_AMOC/4.1)_svm_predict_gcm_direct_ba/`) and the **chained** method (20CR-solution,
`4_SVM_predict_AMOC/4.2)_svm_predict_gcm_chained_ba/`). **ISIMIP is deliberately excluded here** --
it is a regular SSP585 projection, not part of the AMOC-collapse comparison this stage is for.

**Both input levels, High and Low** (2026-10-14, per discussion -- the old single-mask design
had no level distinction at all). `soil_suitability_<crop>_1deg_x0grid_<high/low>.nc` gives,
per crop/level and 1° cell, a continuous SSR index (0-100, rescaled to 0-1 here). Used in two
ways: **areas (Mha)** = climate-suitable cell area x index (nothing thrown away); **change-map
categories** need a binary layer, so `index >= TAU_MAP` (0.4, same cutoff as `THRESH_BIN` in
`2_soil_suitability_gaez/2.d)` -- the mS/MS boundary GAEZ itself uses between marginal and
genuinely cultivable land). A cutoff sensitivity (0.25/0.4/0.5/0.75) is reported in the summary
table. Every output is now duplicated per level (filenames suffixed `_high`/`_low`); nothing
from the High-input run overwrites the Low-input run or vice versa.

**Input**: every `suitability_*.nc` already produced by `4.1)_svm_predict_gcm_direct_ba.ipynb`
and `4.2)_svm_predict_gcm_chained_ba.ipynb`, plus
`1_SVM/2_soil_suitability_gaez/soil_suitability_outputs/soil_suitability_<crop>_1deg_x0grid_<level>.nc`.
**Output**: one companion NetCDF per input file per source per level (same base filename +
level suffix, original variables preserved + new `_soil` variables), plus soil-masked change
maps and country analysis PNGs, under `svm_predict_outputs_ECearthdirectsolution_soil/` and
`svm_predict_outputs_20CRsolution_1deg_soil/` in this folder. Nothing in `4_SVM_predict_AMOC` is
touched or overwritten.

In [1]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import regionmask
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE     = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
SOIL_DIR = BASE / "1_SVM" / "2_soil_suitability_gaez" / "soil_suitability_outputs"
THIS_DIR = BASE / "1_SVM" / "6_Suitability_climate_soil_combined_AMOC"

LEVELS   = ['high', 'low']
TAU_MAP  = 0.4  # SSR >= 0.4 (on the 0-1 rescaled index) = "suitable", same cutoff as THRESH_BIN=40
                # in 2_soil_suitability_gaez/2.d) -- GAEZ's own mS/MS boundary -- and numerically
                # identical to the old 1_SVM/2_soil_suitability module's tau_map=0.4.

# Both AMOC-hosing methods this stage covers -- ISIMIP is not part of this comparison (see markdown above).
SOURCES = {
    'ECEarth_direct': dict(
        stage4_dir=BASE / "1_SVM" / "4_SVM_predict_AMOC" / "4.1)_svm_predict_gcm_direct_ba" / "svm_predict_outputs_direct_BA",  # TODO : sorties des notebooks AMOC 4.1/4.2, non copies dans ce dossier
        out_dir=THIS_DIR / "svm_predict_outputs_ECearthdirectsolution_soil",
        norway_pattern='suitability_{crop}_ECEarth_2025_hosing.nc'),
    '20CR_chained': dict(
        stage4_dir=BASE / "1_SVM" / "4_SVM_predict_AMOC" / "4.2)_svm_predict_gcm_chained_ba" / "svm_predict_outputs_chained_BA",  # TODO : sorties des notebooks AMOC 4.1/4.2, non copies dans ce dossier
        out_dir=THIS_DIR / "svm_predict_outputs_20CRsolution_1deg_soil",
        norway_pattern='suitability_{crop}_2025_hosing_With20CRsolution.nc'),
}
for cfg in SOURCES.values():
    cfg['out_dir'].mkdir(parents=True, exist_ok=True)

CROPS     = ['Maize_rf', 'Wheat-spring_rf', 'Wheat-winter_rf']
COUNTRIES = ['France', 'Spain', 'Italy', 'Germany', 'Netherlands']

print("Soil suitability files (GAEZ Module IV, 2_soil_suitability_gaez):")
for level in LEVELS:
    for crop in CROPS:
        p = SOIL_DIR / f'soil_suitability_{crop}_1deg_x0grid_{level}.nc'
        print(f'  {"OK" if p.exists() else "MISSING"}: {p.name}')
for name, cfg in SOURCES.items():
    files = sorted(cfg['stage4_dir'].glob('suitability_*.nc'))
    cfg['files'] = files
    print(f'{name}: {"OK" if cfg["stage4_dir"].exists() else "MISSING"} {cfg["stage4_dir"]} -- {len(files)} files')

Soil suitability files (GAEZ Module IV, 2_soil_suitability_gaez):
  OK: soil_suitability_Maize_rf_1deg_x0grid_high.nc
  OK: soil_suitability_Wheat-spring_rf_1deg_x0grid_high.nc
  OK: soil_suitability_Wheat-winter_rf_1deg_x0grid_high.nc
  OK: soil_suitability_Maize_rf_1deg_x0grid_low.nc
  OK: soil_suitability_Wheat-spring_rf_1deg_x0grid_low.nc
  OK: soil_suitability_Wheat-winter_rf_1deg_x0grid_low.nc
ECEarth_direct: OK C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_codes\crop_suitability_code_bdap\1_SVM\4_SVM_predict\svm_predict_outputs_direct_BA -- 6 files
20CR_chained: OK C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_codes\crop_suitability_code_bdap\1_SVM\4_SVM_predict\svm_predict_outputs_chained_BA -- 6 files


C:\Users\elisa\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load soil mask + define the AND-combine / country-analysis pipeline

`suit_ref_bin`/`suit_fut_bin` are not persisted in the stage-4 files, only `change_cat`
is — reconstructed here from `change_cat` (`0=stays unsuit,1=gains,2=stays suit,3=loses`)
rather than re-running the SVM. Areas and country analysis are weighted by the
soil-suitability index; the change-map categories use the binary layer (index ≥
`tau_map`). Country-area cells are the same `regionmask` + `cos(lat)`-weighted-area
pattern used in every Stage-4 notebook, just fed the soil-masked bins instead of the
climate-only ones. Identical logic to Stage 2.c/2.d's own soil notebooks before this
merge — only parametrized by `out_dir` so it can run for either source.

In [2]:
# Preload the soil index + binary mask for every (level, crop) -- one small file each,
# cheap to keep all 6 (2 levels x 3 crops) in memory for the rest of the notebook.
soil_data = {}
for level in LEVELS:
    soil_data[level] = {}
    for crop in CROPS:
        ds_s = xr.open_dataset(SOIL_DIR / f'soil_suitability_{crop}_1deg_x0grid_{level}.nc')
        idx = ds_s[list(ds_s.data_vars)[0]].values / 100.0  # 0-100 -> 0-1 (var name doesn't always match crop label -- GAEZ ties wheat-spring/winter to the same generic 'Wheat_rf' soil variable, duplicated per file, not per varname)
        soil_data[level][crop] = dict(idx=idx, bin=(idx >= TAU_MAP).astype(float))
        ds_s.close()
print('soil_data loaded:', {level: list(soil_data[level].keys()) for level in LEVELS})

R_earth = 6371.0
countries_rg = regionmask.defined_regions.natural_earth_v5_0_0.countries_10
name_to_num  = {n: num for n, num in zip(countries_rg.names, countries_rg.numbers)}


def apply_soil_constraint(stage4_path, out_dir, level):
    """
    Load one stage-4 suitability_*.nc, AND-combine its climate-only bins with the
    matching crop's static soil mask (at the given input level), save a companion NetCDF
    (same filename + level suffix, original vars preserved + new _soil vars), and
    regenerate the change-map + country-analysis PNGs from the soil-masked bins. Returns
    a summary dict for the printed overview.
    """
    crop = next(c for c in CROPS if c in stage4_path.name)
    ds4  = xr.open_dataset(stage4_path)
    lats, lons = ds4.lat.values, ds4.lon.values
    nlat, nlon = len(lats), len(lons)

    change_cat = ds4['change_cat'].values
    valid = ~np.isnan(change_cat)
    suit_ref_bin = np.where(valid, (change_cat >= 2).astype(float), np.nan)
    suit_fut_bin = np.where(valid, np.isin(change_cat, [1, 2]).astype(float), np.nan)

    soil_bin  = soil_data[level][crop]['bin']    # index >= TAU_MAP (change-map categories)
    soil_frac = soil_data[level][crop]['idx']    # 0-1 suitable-area fraction (areas)
    suit_ref_bin_soil = suit_ref_bin * soil_bin
    suit_fut_bin_soil = suit_fut_bin * soil_bin

    valid_soil = ~np.isnan(suit_ref_bin_soil) & ~np.isnan(suit_fut_bin_soil)
    change_cat_soil = np.full((nlat, nlon), np.nan)
    change_cat_soil[valid_soil & (suit_ref_bin_soil == 0) & (suit_fut_bin_soil == 0)] = 0
    change_cat_soil[valid_soil & (suit_ref_bin_soil == 0) & (suit_fut_bin_soil == 1)] = 1
    change_cat_soil[valid_soil & (suit_ref_bin_soil == 1) & (suit_fut_bin_soil == 1)] = 2
    change_cat_soil[valid_soil & (suit_ref_bin_soil == 1) & (suit_fut_bin_soil == 0)] = 3

    # --- save companion NetCDF (original vars untouched + new _soil vars) ---
    coords = {'lat': lats, 'lon': lons}
    ds_out = ds4.copy()
    ds_out['soil_suit_idx']      = xr.DataArray(soil_frac, dims=['lat','lon'], coords=coords,
                                                  attrs={'long_name': f'Soil-suitability index (0-1) -- {crop}, {level} input'})
    ds_out['soil_suit_bin']       = xr.DataArray(soil_bin, dims=['lat','lon'], coords=coords,
                                                  attrs={'long_name': f'Soil suitability, index >= TAU_MAP -- {crop}, {level} input',
                                                         'tau_map': float(TAU_MAP)})
    ds_out['suit_ref_bin_soil']   = xr.DataArray(suit_ref_bin_soil, dims=['lat','lon'], coords=coords)
    ds_out['suit_fut_bin_soil']   = xr.DataArray(suit_fut_bin_soil, dims=['lat','lon'], coords=coords)
    ds_out['change_cat_soil']     = xr.DataArray(change_cat_soil, dims=['lat','lon'], coords=coords,
                                                  attrs={'long_name': '0=stays non-suit, 1=gains, 2=stays suit, 3=loses (soil-constrained)'})
    ds_out.attrs['soil_suitability_source'] = str(SOIL_DIR / f'soil_suitability_{crop}_1deg_x0grid_{level}.nc')
    ds_out.attrs['soil_input_level'] = level
    out_nc = out_dir / f'{stage4_path.stem}_{level}{stage4_path.suffix}'
    ds_out.to_netcdf(out_nc)

    # --- change map ---
    ext = [lons[0]-0.5, lons[-1]+0.5, lats[0]-0.5, lats[-1]+0.5]
    cmap_binary = mcolors.ListedColormap(['#d3d3d3', '#4dac26', '#2166ac', '#d7191c'])
    bounds = [-0.5, 0.5, 1.5, 2.5, 3.5]
    norm   = mcolors.BoundaryNorm(bounds, cmap_binary.N)
    tag = stage4_path.stem[len('suitability_'):]

    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(change_cat_soil, origin='lower', aspect='auto', extent=ext, cmap=cmap_binary, norm=norm)
    cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
    cbar.ax.set_yticklabels(['Stays not suitable', 'Gains suitability', 'Stays suitable', 'Loses suitability'])
    ax.set_title(f'Soil-constrained suitability change ({level} input) -- {tag}')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.savefig(out_dir / f'suitability_change_soil_{tag}_{level}.png', dpi=150)
    plt.close(fig)

    # --- before/after comparison: climate-only vs soil-constrained, + detailed status ---
    # The soil mask only ever ANDs suitability away (suit_bin_soil = suit_bin * soil_bin,
    # soil_bin in {0,1}), so a pixel's category can only collapse toward 0 ("stays not
    # suitable") -- it can go 1->0, 2->0 or 3->0, never the other way. Panel 3 is a
    # 7-category map that splits each climate category by whether soil confirmed or
    # overrode it, rather than just showing "what changed" coloured by the old category:
    #   0 = never suitable under climate at all (soil irrelevant)
    #   1/2/3 = real gain/stays-suitable/loses, soil-confirmed (soil_bin=1)
    #   4 = FALSE gain -- climate said newly suitable, soil disqualifies (Norway-type fix)
    #   5 = soil-disqualified despite being climate-suitable in BOTH ref and fut
    #   6 = already losing per climate, and soil also disqualifies (ref forced out too)
    soil_ok = (soil_bin == 1)
    detail = np.full((nlat, nlon), np.nan)
    detail[valid_soil & (change_cat == 0)] = 0
    detail[valid_soil & (change_cat == 1) &  soil_ok] = 1
    detail[valid_soil & (change_cat == 2) &  soil_ok] = 2
    detail[valid_soil & (change_cat == 3) &  soil_ok] = 3
    detail[valid_soil & (change_cat == 1) & ~soil_ok] = 4
    detail[valid_soil & (change_cat == 2) & ~soil_ok] = 5
    detail[valid_soil & (change_cat == 3) & ~soil_ok] = 6

    cmap_detail = mcolors.ListedColormap(
        ['#d3d3d3', '#4dac26', '#2166ac', '#d7191c', '#fdae61', '#984ea3', '#800026'])
    bounds_detail = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5]
    norm_detail   = mcolors.BoundaryNorm(bounds_detail, cmap_detail.N)
    detail_labels = [
        'Never suitable (climate)', 'Real gain (soil-confirmed)', 'Real stays-suitable (soil-confirmed)',
        'Real loss (soil-confirmed)', 'FALSE gain -- corrected by soil',
        'Soil-disqualified (climate-suitable both periods)', 'Loss reinforced by soil',
    ]

    fig, axes = plt.subplots(1, 3, figsize=(26, 7))
    im0 = axes[0].imshow(change_cat, origin='lower', aspect='auto', extent=ext, cmap=cmap_binary, norm=norm)
    axes[0].set_title(f'Climate-only (before soil)\n{tag}')
    cbar0 = plt.colorbar(im0, ax=axes[0], ticks=[0, 1, 2, 3], shrink=0.8)
    cbar0.ax.set_yticklabels(['Stays not suitable', 'Gains suitability', 'Stays suitable', 'Loses suitability'], fontsize=8)

    im1 = axes[1].imshow(change_cat_soil, origin='lower', aspect='auto', extent=ext, cmap=cmap_binary, norm=norm)
    axes[1].set_title(f'Climate + soil-constrained ({level} input)\n{tag}')
    cbar1 = plt.colorbar(im1, ax=axes[1], ticks=[0, 1, 2, 3], shrink=0.8)
    cbar1.ax.set_yticklabels(['Stays not suitable', 'Gains suitability', 'Stays suitable', 'Loses suitability'], fontsize=8)

    im2 = axes[2].imshow(detail, origin='lower', aspect='auto', extent=ext, cmap=cmap_detail, norm=norm_detail)
    axes[2].set_title(f'Detailed status (climate x soil, {level} input)\n{tag}')
    cbar2 = plt.colorbar(im2, ax=axes[2], ticks=list(range(7)), shrink=0.8)
    cbar2.ax.set_yticklabels(detail_labels, fontsize=7)

    for ax in axes:
        ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.savefig(out_dir / f'suitability_change_comparison_{tag}_{level}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    n_never      = int(np.nansum(detail == 0))
    n_false_gain = int(np.nansum(detail == 4))
    n_disq_suit  = int(np.nansum(detail == 5))
    n_disq_loss  = int(np.nansum(detail == 6))
    n_changed    = n_false_gain + n_disq_suit + n_disq_loss
    print(f'  {tag} ({level}): soil constraint changed {n_changed} pixels '
          f'({n_false_gain} false gains corrected, {n_disq_suit} always-climate-suitable-but-soil-disqualified, '
          f'{n_disq_loss} already-losing-and-soil-disqualified); {n_never} pixels never suitable under climate at all')

    # --- area + country analysis (soil-masked bins) ---
    lat_area_km2   = (np.pi/180*R_earth)**2 * np.cos(np.radians(lats))
    pixel_area_Mha = np.tile(lat_area_km2[:, None], (1, nlon)) * 100 / 1e6

    def suitable_area(bin_arr, area, extra_mask=None):
        m = (bin_arr == 1) & ~np.isnan(bin_arr)
        if extra_mask is not None:
            m = m & extra_mask
        return float(np.nansum(area[m]))

    def weighted_area(clim_bin, frac, area, extra_mask=None):
        """Climate-suitable cell area x soil-suitable fraction (nothing thrown away)."""
        m = (clim_bin == 1) & ~np.isnan(clim_bin) & ~np.isnan(frac)
        if extra_mask is not None:
            m = m & extra_mask
        return float(np.sum(area[m] * frac[m]))

    area_ref = weighted_area(suit_ref_bin, soil_frac, pixel_area_Mha)
    area_fut = weighted_area(suit_fut_bin, soil_frac, pixel_area_Mha)
    area_ref_bin = suitable_area(suit_ref_bin_soil, pixel_area_Mha)
    area_fut_bin = suitable_area(suit_fut_bin_soil, pixel_area_Mha)
    sens = {f'area_fut_Mha_bin_tau{int(round(t*100))}':
            suitable_area(suit_fut_bin * (soil_frac >= t), pixel_area_Mha) for t in (0.25, 0.4, 0.5, 0.75)}

    mask_da = countries_rg.mask(lons, lats)
    rows = []
    for name in COUNTRIES:
        c_num = name_to_num.get(name)
        if c_num is None:
            continue
        c_mask = (mask_da.values == c_num)
        a_ref = weighted_area(suit_ref_bin, soil_frac, pixel_area_Mha, c_mask)
        a_fut = weighted_area(suit_fut_bin, soil_frac, pixel_area_Mha, c_mask)
        rows.append({'Country': name, 'Ref_Mha': a_ref, 'Fut_Mha': a_fut,
                     'Change_Mha': a_fut - a_ref,
                     'Change_pct': 100*(a_fut-a_ref)/a_ref if a_ref > 0 else 0.0})
    df_c = pd.DataFrame(rows)

    colors = ['#d7191c' if x < 0 else '#4dac26' for x in df_c['Change_pct']]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh(df_c['Country'], df_c['Change_Mha'], color=colors)
    axes[0].axvline(0, color='black', linewidth=0.8)
    axes[0].set_xlabel('Change (Mha)'); axes[0].set_title(f'Absolute change (soil-weighted, {level} input) -- {tag}')
    axes[1].barh(df_c['Country'], df_c['Change_pct'], color=colors)
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].set_xlabel('Change (%)'); axes[1].set_title(f'Relative change (soil-weighted, {level} input) -- {tag}')
    plt.tight_layout()
    plt.savefig(out_dir / f'country_analysis_soil_{tag}_{level}.png', dpi=150)
    plt.close(fig)

    ds4.close()
    return {
        'source': None, 'file': stage4_path.name, 'crop': crop, 'level': level,
        'area_ref_Mha_climate_only': float(np.nansum(pixel_area_Mha[(suit_ref_bin==1)])),
        'area_fut_Mha_climate_only': float(np.nansum(pixel_area_Mha[(suit_fut_bin==1)])),
        'area_ref_Mha_soil_weighted': area_ref,
        'area_fut_Mha_soil_weighted': area_fut,
        'area_ref_Mha_soil_bin': area_ref_bin,
        'area_fut_Mha_soil_bin': area_fut_bin,
        **sens,
        'n_pixels_changed_by_soil': n_changed,
    }

print("apply_soil_constraint() defined (level-aware).")

soil_data loaded: {'high': ['Maize_rf', 'Wheat-spring_rf', 'Wheat-winter_rf'], 'low': ['Maize_rf', 'Wheat-spring_rf', 'Wheat-winter_rf']}
apply_soil_constraint() defined (level-aware).


## 2. Run for every Stage-4 file found on disk, both sources

In [3]:
all_summaries = []
for level in LEVELS:
    for source_name, cfg in SOURCES.items():
        print(f'--- {source_name} ({level} input) ---')
        for p in cfg["files"]:
            s = apply_soil_constraint(p, cfg["out_dir"], level)
            s["source"] = source_name
            all_summaries.append(s)

df_summary = pd.DataFrame(all_summaries)
df_summary["area_fut_reduction_Mha"] = (df_summary["area_fut_Mha_climate_only"]
                                         - df_summary["area_fut_Mha_soil_weighted"])
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print(df_summary.drop(columns=["file"]).round(1).to_string(index=False))
for name, cfg in SOURCES.items():
    print(f"Saved {len(cfg['files'])*len(LEVELS)} companion NetCDFs + PNGs ({len(LEVELS)} levels) to: {cfg['out_dir']}")

--- ECEarth_direct (high input) ---


  Maize_rf_ECEarth_2025_hosing (high): soil constraint changed 116 pixels (7 false gains corrected, 91 always-climate-suitable-but-soil-disqualified, 18 already-losing-and-soil-disqualified); 484 pixels never suitable under climate at all


  Maize_rf_ECEarth_2025_nohosing (high): soil constraint changed 151 pixels (42 false gains corrected, 87 always-climate-suitable-but-soil-disqualified, 22 already-losing-and-soil-disqualified); 376 pixels never suitable under climate at all


  Wheat-spring_rf_ECEarth_2025_hosing (high): soil constraint changed 150 pixels (70 false gains corrected, 80 always-climate-suitable-but-soil-disqualified, 0 already-losing-and-soil-disqualified); 135 pixels never suitable under climate at all


  Wheat-spring_rf_ECEarth_2025_nohosing (high): soil constraint changed 167 pixels (87 false gains corrected, 77 always-climate-suitable-but-soil-disqualified, 3 already-losing-and-soil-disqualified); 112 pixels never suitable under climate at all


...

  Wheat-spring_rf_2025_hosing_With20CRsolution (low): soil constraint changed 377 pixels (200 false gains corrected, 161 always-climate-suitable-but-soil-disqualified, 16 already-losing-and-soil-disqualified); 94 pixels never suitable under climate at all


  Wheat-spring_rf_2025_nohosing_With20CRsolution (low): soil constraint changed 402 pixels (225 false gains corrected, 164 always-climate-suitable-but-soil-disqualified, 13 already-losing-and-soil-disqualified); 61 pixels never suitable under climate at all


  Wheat-winter_rf_2025_hosing_With20CRsolution (low): soil constraint changed 436 pixels (255 false gains corrected, 179 always-climate-suitable-but-soil-disqualified, 2 already-losing-and-soil-disqualified); 28 pixels never suitable under climate at all


  Wheat-winter_rf_2025_nohosing_With20CRsolution (low): soil constraint changed 444 pixels (263 false gains corrected, 179 always-climate-suitable-but-soil-disqualified, 2 already-losing-and-soil-disqualified); 22 pixels never suitable under climate at all


        source            crop level  area_ref_Mha_climate_only  area_fut_Mha_climate_only  area_ref_Mha_soil_weighted  area_fut_Mha_soil_weighted  area_ref_Mha_soil_bin  area_fut_Mha_soil_bin  area_fut_Mha_bin_tau25  area_fut_Mha_bin_tau40  area_fut_Mha_bin_tau50  area_fut_Mha_bin_tau75  n_pixels_changed_by_soil  area_fut_reduction_Mha
ECEarth_direct        Maize_rf  high                      439.5                      523.1                       246.8                       323.7                  338.1                  433.7                   511.8                   433.7                   343.7                   153.4                       116                   199.4
ECEarth_direct        Maize_rf  high                      439.5                      590.3                       246.8                       360.1                  338.1                  480.6                   572.9                   480.6                   383.7                   166.0                       151        

## 3. Motivating-bug check — Norway under `2025_hosing`, both sources

Direct check of the observation that started this whole pipeline stage: under the
climate-only prediction, does any Norwegian pixel flip from unsuitable to "gains
suitability" (`change_cat==1`)? If so, does the soil mask correctly pull it back to
"stays not suitable" (`change_cat_soil==0`) once soil is accounted for? Checked for both
the direct (EC-Earth3) and chained (20CR-solution) AMOC-hosing methods, both input levels.

In [4]:
norway_num = name_to_num.get("Norway")

for level in LEVELS:
    for source_name, cfg in SOURCES.items():
        print(f"=== {source_name} ({level} input) ===")
        for crop in ["Maize_rf", "Wheat-spring_rf", "Wheat-winter_rf"]:
            stage4_path = cfg["stage4_dir"] / cfg["norway_pattern"].format(crop=crop)
            soil_path   = cfg["out_dir"] / f"{stage4_path.stem}_{level}{stage4_path.suffix}"
            if not stage4_path.exists() or not soil_path.exists():
                print(f"{crop}: file not found, skipping"); continue

            ds4   = xr.open_dataset(stage4_path)
            dsoil = xr.open_dataset(soil_path)
            lats, lons = ds4.lat.values, ds4.lon.values
            norway_mask = (countries_rg.mask(lons, lats).values == norway_num)

            gained = (ds4["change_cat"].values == 1) & norway_mask
            n_gained = int(gained.sum())
            if n_gained == 0:
                print(f"{crop:18s}: no Norwegian pixel gains climate suitability under 2025_hosing -- nothing to correct here")
                ds4.close(); dsoil.close()
                continue

            still_unsuit_after_soil = int(((dsoil["change_cat_soil"].values == 0) & gained).sum())
            print(f"{crop:18s}: {n_gained} Norwegian pixel(s) gain CLIMATE suitability under 2025_hosing "
                  f"-> {still_unsuit_after_soil}/{n_gained} correctly pulled back to not suitable by the soil mask")
            ds4.close(); dsoil.close()

=== ECEarth_direct (high input) ===


Maize_rf          : no Norwegian pixel gains climate suitability under 2025_hosing -- nothing to correct here


Wheat-spring_rf   : 20 Norwegian pixel(s) gain CLIMATE suitability under 2025_hosing -> 8/20 correctly pulled back to not suitable by the soil mask


Wheat-winter_rf   : 23 Norwegian pixel(s) gain CLIMATE suitability under 2025_hosing -> 13/23 correctly pulled back to not suitable by the soil mask
=== 20CR_chained (high input) ===


Maize_rf          : 5 Norwegian pixel(s) gain CLIMATE suitability under 2025_hosing -> 3/5 correctly pulled back to not suitable by the soil mask


...

Wheat-spring_rf   : 20 Norwegian pixel(s) gain CLIMATE suitability under 2025_hosing -> 20/20 correctly pulled back to not suitable by the soil mask


Wheat-winter_rf   : 23 Norwegian pixel(s) gain CLIMATE suitability under 2025_hosing -> 23/23 correctly pulled back to not suitable by the soil mask
=== 20CR_chained (low input) ===


Maize_rf          : 5 Norwegian pixel(s) gain CLIMATE suitability under 2025_hosing -> 5/5 correctly pulled back to not suitable by the soil mask


Wheat-spring_rf   : 53 Norwegian pixel(s) gain CLIMATE suitability under 2025_hosing -> 53/53 correctly pulled back to not suitable by the soil mask


Wheat-winter_rf   : 55 Norwegian pixel(s) gain CLIMATE suitability under 2025_hosing -> 55/55 correctly pulled back to not suitable by the soil mask
